# Forecast calibration

A forecast is *calibrated* if, when it says P(up)=0.6, the asset actually rises ~60% of
the time. This notebook bins historical forecasts by predicted `p_up` and compares to the
realized up-rate over the forecast horizon.

Calibration is a prerequisite for trusting the edge score. **Mock forecasts are not
expected to be calibrated** — run this against real-model forecasts.

In [ ]:
import os, sys
os.environ.setdefault('KAT_MOCK_MODE', 'true')
sys.path.insert(0, os.path.abspath('../backend'))
import numpy as np, pandas as pd
from app.config import get_config_store
from app.db.session import new_session
from app.repositories import candles_repo
from app.kronos.adapter import KronosAdapter
store = get_config_store(); k = store.kronos()

In [ ]:
# Roll through history: forecast at t (past-only), record predicted p_up and realized outcome.
db = new_session(); symbol, tf = 'BTC/USDT', '1h'
rows = candles_repo.get_candles(db, symbol, tf, ascending=True, limit=4000)
df = candles_repo.candles_to_df(rows)
adapter = KronosAdapter(k)
ctx, hz = k.context_length, k.forecast_horizon
records = []
for i in range(ctx, len(df) - hz, 12):  # step to keep it light
    hist = df.iloc[:i]
    dist = adapter.forecast(hist, symbol=symbol, timeframe=tf)
    realized_up = df['close'].iloc[i + hz - 1] > df['close'].iloc[i - 1]
    records.append({'p_up': dist.p_up, 'realized_up': bool(realized_up)})
cal = pd.DataFrame(records)
print(len(cal), 'forecasts')

In [ ]:
import matplotlib.pyplot as plt
cal['bin'] = pd.cut(cal['p_up'], bins=np.linspace(0, 1, 11))
grp = cal.groupby('bin', observed=True).agg(pred=('p_up','mean'), real=('realized_up','mean'), n=('p_up','size')).dropna()
plt.plot([0,1],[0,1],'--',c='gray',label='perfect')
plt.plot(grp['pred'], grp['real'], 'o-', label='observed')
plt.xlabel('predicted P(up)'); plt.ylabel('realized up-rate'); plt.legend(); plt.title('Reliability diagram'); plt.show()
grp
db.close()